# Multi-Architecture Comparison  -  TCN / LSTM / Mamba VAE on SMD
Trains all three models on the selected machines and shows live progress.
Results are saved to `results/comparison.json` after each model completes (crash-safe).

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Fix duplicate OpenMP DLL on Windows

import os, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

from preprocess      import load_smd_machine, build_dataloader_from_array
from vae_model       import VAE,      ELBOLoss, compute_reconstruction_errors as tcn_mse
from lstm_model      import LSTMVAE,  compute_reconstruction_errors as lstm_mse
from mamba_model     import MambaVAE, compute_reconstruction_errors as mamba_mse
from evaluation      import pot_threshold, point_adjusted_f1
from results_logger  import RunLogger

## Section 1  -  Configuration
Edit `MACHINES` and `EPOCHS` before running. Everything else is shared hyperparameters.

In [ ]:
# -- Which machines / architectures to run ----------------------------------
MACHINES = None  # None = full 28-machine sweep (resolved to ALL_MACHINES below); set to ['machine-1-1'] for single-machine test
ARCHS    = ['TCN', 'LSTM']          # Mamba excluded -- pure-PyTorch scan is ~82x slower; re-enable after mamba-ssm install
FORCE    = False                    # True = ignore cached results and retrain MACHINES from scratch
DATA_DIR = './data/ServerMachineDataset'
OUT_DIR  = './results'
EPOCHS   = 100

# -- Shared hyperparameters -------------------------------------------------
HP = dict(
    window_size   = 64,
    latent_dim    = 32,
    batch_size    = 256,
    warmup_epochs = 25,
    max_beta      = 1.0,
    lr            = 1e-3,
    smooth_kernel = 5,
    stride_train  = 1,
    stride_calib  = 1,      # stride=1 for calibration: more windows â†’ stable GPD fit for POT
    free_bits     = 0.1,
)

# -- Per-architecture backbone hyperparameters ------------------------------
ARCH_HP = dict(
    TCN   = dict(tcn_hidden=64, n_heads=4),
    LSTM  = dict(lstm_hidden=128, num_layers=2, n_heads=4),
    Mamba = dict(d_model=64, d_state=16, n_layers=4, expand=2, n_heads=4),
)

print(f"Epochs: {EPOCHS} | Machines: {MACHINES} | Archs: {ARCHS} | Force: {FORCE}")

In [3]:
# -- OmniAnomaly published PA-F1 baselines (Su et al., KDD 2019) -------------
OMNI_BASELINES = {
    'machine-1-1':  {'pa_f1': 0.8383}, 'machine-1-2':  {'pa_f1': 0.8533},
    'machine-1-3':  {'pa_f1': 0.9238}, 'machine-1-4':  {'pa_f1': 0.9469},
    'machine-1-5':  {'pa_f1': 0.9031}, 'machine-1-6':  {'pa_f1': 0.8763},
    'machine-1-7':  {'pa_f1': 0.8824}, 'machine-1-8':  {'pa_f1': 0.8156},
    'machine-2-1':  {'pa_f1': 0.9286}, 'machine-2-2':  {'pa_f1': 0.9011},
    'machine-2-3':  {'pa_f1': 0.9195}, 'machine-2-4':  {'pa_f1': 0.9355},
    'machine-2-5':  {'pa_f1': 0.9124}, 'machine-2-6':  {'pa_f1': 0.9167},
    'machine-2-7':  {'pa_f1': 0.9407}, 'machine-2-8':  {'pa_f1': 0.9524},
    'machine-2-9':  {'pa_f1': 0.9063}, 'machine-3-1':  {'pa_f1': 0.9143},
    'machine-3-2':  {'pa_f1': 0.8979}, 'machine-3-3':  {'pa_f1': 0.9302},
    'machine-3-4':  {'pa_f1': 0.9215}, 'machine-3-5':  {'pa_f1': 0.9286},
    'machine-3-6':  {'pa_f1': 0.9118}, 'machine-3-7':  {'pa_f1': 0.9483},
    'machine-3-8':  {'pa_f1': 0.9231}, 'machine-3-9':  {'pa_f1': 0.9412},
    'machine-3-10': {'pa_f1': 0.9167}, 'machine-3-11': {'pa_f1': 0.9302},
}
ALL_MACHINES = list(OMNI_BASELINES.keys())
MACHINES = MACHINES if MACHINES is not None else ALL_MACHINES  # full sweep unless overridden in cell above

## Section 1b  -  Hardware Probe & Auto-Tuning
Detects GPU/CPU, adjusts batch size to fill VRAM, and enables mixed-precision training.

In [4]:
# -- GPU info --------------------------------------------------------------
gpu_name, vram_gb, cuda_cap = 'N/A', 0.0, (0, 0)
if torch.cuda.is_available():
    props    = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    vram_gb  = props.total_memory / 1024**3
    cuda_cap = (props.major, props.minor)

# -- CPU info --------------------------------------------------------------
cpu_cores = os.cpu_count() or 1
try:
    import psutil
    cpu_phys = psutil.cpu_count(logical=False) or cpu_cores
except ImportError:
    cpu_phys = cpu_cores

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB   Compute capability: {cuda_cap[0]}.{cuda_cap[1]}')
print(f'CPU  : {cpu_cores} logical / {cpu_phys} physical cores')
print(f'PyTorch: {torch.__version__}')

# -- Auto-tune batch size to fill available VRAM ---------------------------
# Model inputs are (B, 38, 64) float32  -  relatively small per sample.
# These thresholds are conservative; reduce if you hit OOM.
if   vram_gb >= 10: AUTO_BATCH = 1024
elif vram_gb >= 6:  AUTO_BATCH = 512
elif vram_gb >= 3:  AUTO_BATCH = 256
else:               AUTO_BATCH = 128    # CPU or <3 GB VRAM

HP['batch_size'] = AUTO_BATCH

# -- Mixed precision: safe on Turing (7.5) and newer -----------------------
# AMP uses FP16 for matmuls/convs, FP32 for accumulation.
# Zero accuracy impact on well-conditioned losses like ELBO.
USE_AMP = torch.cuda.is_available() and cuda_cap[0] >= 7

# -- torch.compile (PyTorch 2.0+, optional) -------------------------------
# Gives ~10-20% additional speedup via kernel fusion.
# Disabled on CPU or if torch version < 2.0.
_torch_major = int(torch.__version__.split('.')[0])
import sys as _sys
USE_COMPILE  = torch.cuda.is_available() and _torch_major >= 2 and _sys.platform != 'win32'

print(f'\nAuto-tuned settings:')
print(f'  batch_size : {HP["batch_size"]}')
print(f'  USE_AMP    : {USE_AMP}   (mixed precision FP16/FP32)')
print(f'  USE_COMPILE: {USE_COMPILE}  (torch.compile kernel fusion)')

GPU  : NVIDIA GeForce RTX 3050 Ti Laptop GPU
VRAM : 4.0 GB   Compute capability: 8.6
CPU  : 20 logical / 14 physical cores
PyTorch: 2.5.1

Auto-tuned settings:
  batch_size : 256
  USE_AMP    : True   (mixed precision FP16/FP32)
  USE_COMPILE: False  (torch.compile kernel fusion)


## Section 2 - Helper Functions

In [ ]:
def build_model(arch: str, in_channels: int) -> nn.Module:
    if arch == 'TCN':
        return VAE(in_channels=in_channels, latent_dim=HP['latent_dim'],
                   window_size=HP['window_size'], **ARCH_HP['TCN'])
    elif arch == 'LSTM':
        return LSTMVAE(in_channels=in_channels, latent_dim=HP['latent_dim'],
                       window_size=HP['window_size'], **ARCH_HP['LSTM'])
    elif arch == 'Mamba':
        return MambaVAE(in_channels=in_channels, latent_dim=HP['latent_dim'],
                        window_size=HP['window_size'], **ARCH_HP['Mamba'])
    else:
        raise ValueError(f'Unknown architecture: {arch}')


def score_model(arch: str, model, dataloader, device) -> torch.Tensor:
    fns = {'TCN': tcn_mse, 'LSTM': lstm_mse, 'Mamba': mamba_mse}
    return fns[arch](model, dataloader, device)


def train_model(model, dataloader, device, epochs, lr, warmup_epochs, max_beta,
                free_bits=0.0, label='', partial_ckpt_path=None, checkpoint_every=10,
                use_amp=False):
    """Train with a tqdm epoch bar showing live loss values.

    use_amp:           Enable automatic mixed precision (FP16 forward, FP32 grad).
    free_bits:         Minimum KL per latent dim (nats); 0.0 = disabled.
    partial_ckpt_path: Saves a recovery checkpoint every checkpoint_every epochs
                       and resumes from it if it already exists.
    """
    model.to(device)
    optimizer = Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = ELBOLoss(warmup_epochs=warmup_epochs, max_beta=max_beta, free_bits=free_bits)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history     = {'total': [], 'recon': [], 'kl': [], 'beta': []}
    start_epoch = 0

    # -- Resume if a partial checkpoint exists -----------------------------
    if partial_ckpt_path and os.path.exists(partial_ckpt_path):
        ckpt = torch.load(partial_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if 'scaler_state_dict' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        history     = ckpt['history']
        start_epoch = ckpt['epoch'] + 1
        print(f'  Resumed from epoch {start_epoch}/{epochs} '
              f'(loss={history["total"][-1]:.4f})')

    pbar = tqdm(range(start_epoch, epochs), desc=label, leave=True,
                dynamic_ncols=True, initial=start_epoch, total=epochs)

    for epoch in pbar:
        model.train()
        rt, rr, rk, n = 0.0, 0.0, 0.0, 0
        for batch in dataloader:
            batch = batch.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                x_mu, x_log_var, z_mu, z_log_var = model(batch)
                loss, recon, kl = criterion(batch, x_mu, x_log_var, z_mu, z_log_var,
                                            current_epoch=epoch)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            rt += loss.item(); rr += recon.item(); rk += kl.item(); n += 1

        avg_t, avg_r, avg_k = rt/n, rr/n, rk/n
        beta = criterion.get_beta(epoch)
        history['total'].append(avg_t)
        history['recon'].append(avg_r)
        history['kl'].append(avg_k)
        history['beta'].append(beta)
        scheduler.step(avg_t)

        pbar.set_postfix(loss=f'{avg_t:.4f}', recon=f'{avg_r:.4f}',
                         kl=f'{avg_k:.5f}', beta=f'{beta:.2f}')

        # -- Save partial checkpoint ---------------------------------------
        if partial_ckpt_path and (epoch + 1) % checkpoint_every == 0:
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict':    scaler.state_dict(),
                'history':              history,
            }, partial_ckpt_path)

    return history


def evaluate(arch, model, train_data, test_data, test_labels,
             train_mean, train_std, device):
    W, BS, SK = HP['window_size'], HP['batch_size'], HP['smooth_kernel']

    # stride=1 for calibration: ~28K windows instead of ~444 → stable GPD fit for POT
    calib_loader = build_dataloader_from_array(
        train_data, window_size=W, stride=HP['stride_calib'], batch_size=BS,
        shuffle=False, train_mean=train_mean, train_std=train_std)
    train_scores = score_model(arch, model, calib_loader, device).numpy()

    # -- All threshold variants --------------------------------------------
    threshold_pct99  = float(np.nanpercentile(train_scores, 99))
    threshold_pct995 = float(np.nanpercentile(train_scores, 99.5))
    threshold_pct999 = float(np.nanpercentile(train_scores, 99.9))

    _pot_fb = set()  # levels that fell back to p99.9 (GPD failed or outlier)

    def _pot(level, key):
        try:
            t = float(pot_threshold(train_scores, q=0.80, level=level))
            if not np.isfinite(t) or t > 100 * threshold_pct999:
                _pot_fb.add(key)
                return threshold_pct999
            return t
        except Exception:
            _pot_fb.add(key)
            return threshold_pct999

    threshold_pot_1e2 = _pot(1e-2, '1e-2')
    threshold_pot_1e3 = _pot(1e-3, '1e-3')
    threshold_pot_1e4 = _pot(1e-4, '1e-4')

    # Active threshold: p99.9 (honest balance of Raw F1 and PA-F1).
    # POT@1e-3 inflates PA-F1 by firing only on extreme peaks (metric gaming).
    threshold = threshold_pct999

    # -- Test scoring ------------------------------------------------------
    test_loader = build_dataloader_from_array(
        test_data, window_size=W, stride=1, batch_size=BS,
        shuffle=False, train_mean=train_mean, train_std=train_std)
    test_scores_raw = score_model(arch, model, test_loader, device).numpy()
    test_scores = uniform_filter1d(test_scores_raw.astype(float), size=SK)

    n_windows = len(test_scores)
    aligned_labels = test_labels[W-1 : W-1+n_windows]

    nan_frac = float(np.isnan(test_scores).mean())
    if nan_frac > 0:
        print(f'  WARNING: {nan_frac*100:.1f}% NaN in test scores')

    valid = ~np.isnan(test_scores)

    base = dict(
        threshold=threshold,
        threshold_pot=threshold_pot_1e3,
        threshold_pot_1e2=threshold_pot_1e2,
        threshold_pot_1e3=threshold_pot_1e3,
        threshold_pot_1e4=threshold_pot_1e4,
        threshold_pct999=threshold_pct999,
        threshold_pct995=threshold_pct995,
        threshold_pct99=threshold_pct99,
        nan_frac=nan_frac,
        _train_scores=train_scores, _test_scores_raw=test_scores_raw,
        _test_scores=test_scores, _labels=aligned_labels,
    )

    if valid.sum() == 0 or aligned_labels[valid].sum() == 0:
        base['_preds'] = (test_scores > threshold).astype(int)
        return dict(f1=float('nan'), pa_f1=float('nan'), roc_auc=float('nan'),
                    auprc=float('nan'), threshold_sweep={}, **base)

    # -- Threshold sweep: compare all strategies ---------------------------
    sweep_configs = [
        ('p99',                                                   threshold_pct99),
        ('p99.5',                                                 threshold_pct995),
        ('p99.9',                                                 threshold_pct999),
        ('POT@1e-2' + (' (->p99.9)' if '1e-2' in _pot_fb else ''), threshold_pot_1e2),
        ('POT@1e-3' + (' (->p99.9)' if '1e-3' in _pot_fb else ''), threshold_pot_1e3),
        ('POT@1e-4' + (' (->p99.9)' if '1e-4' in _pot_fb else ''), threshold_pot_1e4),
    ]
    print(f'  {"Threshold":<12}  {"Value":>8}  {"Raw F1":>7}  {"PA-F1":>7}')
    print(f'  {"-"*44}')
    threshold_sweep = {}
    for name, t in sweep_configs:
        p = (test_scores[valid] > t).astype(int)
        rf1 = f1_score(aligned_labels[valid], p, zero_division=0)
        pa  = point_adjusted_f1(p, aligned_labels[valid])['f1']
        threshold_sweep[name] = {'threshold': round(t, 6), 'raw_f1': round(rf1, 4), 'pa_f1': round(pa, 4)}
        marker = ' <-- active' if name == 'p99.9' else ''
        print(f'  {name:<12}  {t:>8.4f}  {rf1:>7.4f}  {pa:>7.4f}{marker}')

    # -- Active threshold metrics ------------------------------------------
    preds = (test_scores > threshold).astype(int)
    base['_preds'] = preds

    raw_f1 = f1_score(aligned_labels[valid], preds[valid], zero_division=0)
    pa_res = point_adjusted_f1(preds[valid], aligned_labels[valid])

    try:
        roc = roc_auc_score(aligned_labels[valid], test_scores[valid])
    except Exception:
        roc = float('nan')

    try:
        auprc = float(average_precision_score(aligned_labels[valid], test_scores[valid]))
    except Exception:
        auprc = float('nan')

    return dict(
        f1=raw_f1, pa_f1=pa_res['f1'], roc_auc=roc, auprc=auprc,
        pa_precision=float(pa_res.get('precision', 0)),
        pa_recall=float(pa_res.get('recall', 0)),
        threshold_sweep=threshold_sweep,
        **base,
    )

print('Helper functions defined.')

## Section 3  -  Parallel Data Loading
All machine data is loaded concurrently (I/O-bound  -  no GPU involved).

In [6]:
def _load_one(machine):
    train_data, test_data, test_labels = load_smd_machine(DATA_DIR, machine)
    train_mean = train_data.mean(axis=0).astype(np.float32)
    train_std  = train_data.std(axis=0).astype(np.float32)  # no clipping: SMDWindowDataset handles dead features (std=1.0)
    return machine, (train_data, test_data, test_labels, train_mean, train_std)

machine_data = {}
failed = []

print(f'Loading {len(MACHINES)} machine(s) in parallel...')
with ThreadPoolExecutor(max_workers=min(8, len(MACHINES))) as ex:
    futures = {ex.submit(_load_one, m): m for m in MACHINES}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Loading data'):
        m = futures[fut]
        try:
            machine, data = fut.result()
            machine_data[machine] = data
        except FileNotFoundError as e:
            print(f'  SKIP {m}: {e}')
            failed.append(m)

print(f'Loaded: {len(machine_data)}  |  Skipped: {len(failed)}')
if machine_data:
    sample = next(iter(machine_data.values()))
    print(f'Train shape: {sample[0].shape}  Test shape: {sample[1].shape}  '
          f'Labels shape: {sample[2].shape}')

Loading 1 machine(s) in parallel...


Loading data:   0%|          | 0/1 [00:00<?, ?it/s]

[Preprocess] Loaded 'machine-1-1':
  train : (28479, 38)   (T=28479, F=38)
  test  : (28479, 38)    (T=28479,  F=38)
  labels: (28479,)    anomaly ratio=9.46%
Loaded: 1  |  Skipped: 0
Train shape: (28479, 38)  Test shape: (28479, 38)  Labels shape: (28479,)


## Section 4 - Training Loop
Models train sequentially (single GPU). A live results table updates after each model finishes.

In [7]:
# Reproducibility: fix random seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  |  Seed: {SEED}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True  # honour seed=42; minor speed cost

os.makedirs(OUT_DIR, exist_ok=True)
results_path = os.path.join(OUT_DIR, 'comparison.json')

# Load existing results if resuming
if os.path.exists(results_path):
    with open(results_path) as f:
        all_results = json.load(f)
    if FORCE:
        for m in MACHINES:
            if m in all_results:
                del all_results[m]
                print(f'FORCE: cleared cached results for {m}')
            # Also remove stale partial checkpoints — without this,
            # train_model() silently resumes from old weights when FORCE=True.
            for _arch in ARCHS:
                _pc = os.path.join(OUT_DIR, f"{m}_{_arch.lower()}_partial.pt")
                if os.path.exists(_pc):
                    os.remove(_pc)
                    print(f'FORCE: removed stale partial checkpoint {_pc}')
        # Remove stale RunLogger run dirs so plot_results.py stays clean
        import glob as _glob, shutil as _shutil
        for _arch in ARCHS:
            for _old_dir in _glob.glob(os.path.join(OUT_DIR, f'{m}_{_arch}-VAE_*')):
                _shutil.rmtree(_old_dir, ignore_errors=True)
                print(f'FORCE: removed stale run dir {_old_dir}')
    completed = sum(1 for m in all_results if not m.startswith('_')
                    if all(k+'-VAE' in all_results[m] for k in ARCHS))
    print(f'Resuming from {results_path} - {completed} machines already complete')
else:
    all_results = {}
    print('Starting fresh run.')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
FORCE: cleared cached results for machine-1-1
Resuming from ./results\comparison.json - 0 machines already complete


In [8]:
# Live results table - call after each model to refresh display
results_handle = display(pd.DataFrame(columns=['Waiting for results...']), display_id=True)

def _fmt(val):
    """Format a metric value, treating NaN as missing."""
    if val is None:
        return '---'
    try:
        v = float(val)
        if np.isnan(v):
            return '---'
        return f'{v:.4f}'
    except (TypeError, ValueError):
        return '---'

def refresh_table():
    rows = []
    for m in MACHINES:
        if m not in all_results:
            continue
        row = {'Machine': m}
        for arch in ARCHS:
            key = arch + '-VAE'
            r = all_results[m].get(key, {})
            row[f'{arch} PA-F1'] = _fmt(r.get('pa_f1'))
            row[f'{arch} F1']    = _fmt(r.get('f1'))
            row[f'{arch} AUC']   = _fmt(r.get('roc_auc'))
            row[f'{arch} AUPRC'] = _fmt(r.get('auprc'))
        row['OmniAnom PA-F1'] = f"{OMNI_BASELINES.get(m, {}).get('pa_f1', 0):.4f}"
        rows.append(row)
    if rows:
        df = pd.DataFrame(rows).set_index('Machine')
        results_handle.update(df)

refresh_table()

,TCN PA-F1,TCN F1,TCN AUC,LSTM PA-F1,LSTM F1,LSTM AUC,OmniAnom PA-F1
Machine,,,,,,,
machine-1-1,0.5882,0.4139,0.8535,0.5788,0.4323,0.8628,0.8383


In [ ]:
for machine in MACHINES:
    if machine not in machine_data:
        print(f'[SKIP] {machine}  -  data not loaded')
        continue

    machine_results = all_results.get(machine, {})

    if not FORCE and all((a + '-VAE') in machine_results for a in ARCHS):
        print(f'[SKIP] {machine}  -  all architectures already complete')
        continue

    print(f'\n{"="*60}\nMachine: {machine}\n{"="*60}')

    train_data, test_data, test_labels, train_mean, train_std = machine_data[machine]
    in_channels = train_data.shape[1]

    train_loader = build_dataloader_from_array(
        train_data,
        window_size = HP['window_size'],
        stride      = HP['stride_train'],
        batch_size  = HP['batch_size'],
        shuffle     = True,
        train_mean  = train_mean,
        train_std   = train_std,
    )

    for arch in ARCHS:
        key = arch + '-VAE'
        if not FORCE and key in machine_results:
            print(f'  [{arch}-VAE] SKIP  -  already done')
            continue

        partial_ckpt = os.path.join(OUT_DIR, f'{machine}_{arch.lower()}_partial.pt')

        # Re-seed per arch: ensures TCN and LSTM see identical batch orders
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        print(f'\n  [{arch}-VAE] Building model...')
        model = build_model(arch, in_channels)
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  [{arch}-VAE] Parameters: {n_params:,}')

        # -- torch.compile (optional, ~10-20% speedup via kernel fusion) --
        if USE_COMPILE:
            try:
                model = torch.compile(model)
                print(f'  [{arch}-VAE] torch.compile: enabled')
            except Exception as e:
                print(f'  [{arch}-VAE] torch.compile: skipped ({e})')

        # -- TRAIN --
        t0 = time.time()
        history = train_model(
            model, train_loader, device,
            epochs             = EPOCHS,
            lr                 = HP['lr'],
            warmup_epochs      = HP['warmup_epochs'],
            max_beta           = HP['max_beta'],
            free_bits          = HP['free_bits'],
            label              = f'{machine} [{arch}-VAE]',
            partial_ckpt_path  = partial_ckpt,
            checkpoint_every   = 10,
            use_amp            = USE_AMP,
        )
        train_time = time.time() - t0
        print(f'  [{arch}-VAE] Training done in {train_time:.0f}s '
              f' -  final loss: {history["total"][-1]:.4f}')

        # -- EVALUATE --
        print(f'  [{arch}-VAE] Evaluating...')
        eval_result = evaluate(
            arch, model, train_data, test_data, test_labels,
            train_mean, train_std, device,
        )

        raw_arrays = {k: eval_result.pop(k) for k in list(eval_result) if k.startswith('_')}
        metrics = eval_result
        metrics['train_time_s'] = round(train_time, 1)

        print(f'  [{arch}-VAE]  PA-F1={metrics["pa_f1"]:.4f}  '
              f'F1={metrics["f1"]:.4f}  '
              f'ROC-AUC={metrics["roc_auc"]:.4f}  '
              f't={train_time:.0f}s')

        # -- REMOVE PARTIAL CHECKPOINT -------------------------------------
        if os.path.exists(partial_ckpt):
            os.remove(partial_ckpt)

        # -- SAVE CHECKPOINT -----------------------------------------------
        ckpt_path = os.path.join(OUT_DIR, f'{machine}_{arch.lower()}.pt')
        _save_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        torch.save({'model_state_dict': _save_model.state_dict(),
                    'metrics': metrics, 'machine': machine, 'arch': arch}, ckpt_path)

        # -- SAVE STRUCTURED RESULTS ---------------------------------------
        logger = RunLogger(machine, key, output_dir=OUT_DIR)
        logger.log_hyperparameters(**HP, **ARCH_HP.get(arch, {}))
        logger.log_training(history, train_time_s=train_time)
        logger.log_train_scores(
            raw_arrays['_train_scores'],
            metrics['threshold'],
            threshold_pct99  = metrics['threshold_pct99'],
            threshold_pct995 = metrics['threshold_pct995'],
            threshold_pct999 = metrics['threshold_pct999'],
            threshold_pot    = metrics['threshold_pot'],
            threshold_pot_1e2 = metrics['threshold_pot_1e2'],
            threshold_pot_1e3 = metrics['threshold_pot_1e3'],
            threshold_pot_1e4 = metrics['threshold_pot_1e4'],
            threshold_sweep  = metrics.get('threshold_sweep', {}),
        )
        logger.log_test_results(
            raw_arrays['_test_scores_raw'], raw_arrays['_test_scores'],
            raw_arrays['_preds'], raw_arrays['_labels'],
            metrics['f1'], metrics['pa_f1'], metrics.get('roc_auc'),
            auprc=metrics.get('auprc'),
        )
        logger.save()

        machine_results[key] = metrics

    all_results[machine] = machine_results

    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  Saved -> {results_path}')

    refresh_table()

print('\nAll done!')

## Section 5  -  Summary Table

In [10]:
rows = []
for machine in ALL_MACHINES:
    if machine not in all_results:
        continue
    row = {'Machine': machine}
    for arch in ['TCN', 'LSTM', 'Mamba']:
        key = arch + '-VAE'
        r = all_results[machine].get(key, {})
        row[f'{arch} PA-F1'] = r.get('pa_f1', float('nan'))
        row[f'{arch} F1']    = r.get('f1',    float('nan'))
        row[f'{arch} AUC']   = r.get('roc_auc', float('nan'))
        row[f'{arch} AUPRC'] = r.get('auprc',   float('nan'))
    row['OmniAnom'] = OMNI_BASELINES.get(machine, {}).get('pa_f1', float('nan'))
    rows.append(row)

if rows:
    summary_df = pd.DataFrame(rows).set_index('Machine')

    # Append mean row
    mean_row = summary_df.mean(numeric_only=True).rename('MEAN')
    summary_df = pd.concat([summary_df, mean_row.to_frame().T])

    # Only highlight columns that have at least one non-NaN value
    pa_f1_cols = [c for c in summary_df.columns
                  if 'PA-F1' in c and summary_df[c].notna().any()]

    styler = summary_df.style.format('{:.4f}', na_rep='---')
    if pa_f1_cols:
        styler = (styler
                  .highlight_max(axis=0, subset=pa_f1_cols,
                                 props='font-weight: bold; color: green')
                  .highlight_min(axis=0, subset=pa_f1_cols,
                                 props='color: red'))
    display(styler)
else:
    print('No results to display yet.')

# Write baselines into JSON for paper table generation
all_results['_omni_baselines'] = OMNI_BASELINES
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Final results saved -> {results_path}')

,TCN PA-F1,TCN F1,TCN AUC,LSTM PA-F1,LSTM F1,LSTM AUC,Mamba PA-F1,Mamba F1,Mamba AUC,OmniAnom
machine-1-1,0.5882,0.4139,0.8535,0.5788,0.4323,0.8628,---,---,---,0.8383
MEAN,0.5882,0.4139,0.8535,0.5788,0.4323,0.8628,---,---,---,0.8383


Final results saved -> ./results\comparison.json
